In [41]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, csc_matrix
from sklearn.metrics.pairwise import cosine_similarity

In [42]:
df_book_tags = pd.read_csv('archive2/book_tags.csv')
df_books = pd.read_csv('archive2/books.csv')
df_ratings = pd.read_csv('archive2/ratings.csv')
df_tags = pd.read_csv('archive2/tags.csv')
df_to_read = pd.read_csv('archive2/to_read.csv')

In [43]:
title_col = 'title' if 'title' in df_books.columns else 'original_title'
df_books["display_title"] = df_books[title_col].fillna("Unknown")
book_title_dict = df_books.set_index("id")["display_title"].to_dict()
for book_id in df_ratings["book_id"].unique():
    if book_id not in book_title_dict:
        book_title_dict[book_id] = "Unknown"

In [44]:
min_ratings_per_user = 2
user_counts = df_ratings.groupby("user_id").size()
active_users = user_counts[user_counts >= min_ratings_per_user].index
ratings_filtered = df_ratings[df_ratings["user_id"].isin(active_users)].copy()

print(f"Всего пользователей: {df_ratings['user_id'].nunique()}")
print(f"Активных пользователей (>= {min_ratings_per_user} оценок): {len(active_users)}")

Всего пользователей: 53424
Активных пользователей (>= 2 оценок): 53424


In [45]:
user_mean = ratings_filtered.groupby("user_id")["rating"].mean()
ratings_filtered["rating_centered"] = ratings_filtered["rating"] - ratings_filtered["user_id"].map(user_mean)

user_ids = ratings_filtered["user_id"].unique()
book_ids = ratings_filtered["book_id"].unique()

user_to_idx = {user: i for i, user in enumerate(user_ids)}
book_to_idx = {book: j for j, book in enumerate(book_ids)}
idx_to_book = {j: book for book, j in book_to_idx.items()}

rows = ratings_filtered["user_id"].map(user_to_idx)
cols = ratings_filtered["book_id"].map(book_to_idx)
data = ratings_filtered["rating_centered"].astype(np.float32)

user_item_matrix = csr_matrix((data, (rows, cols)), shape=(len(user_ids), len(book_ids)), dtype=np.float32)
print(f"Матрица размером {user_item_matrix.shape}, ненулевых: {user_item_matrix.nnz}")

Матрица размером (53424, 10000), ненулевых: 979478


In [46]:
popular_books = (df_ratings.groupby("book_id").agg(avg_rating=("rating", "mean"), rating_count=("rating", "count"))
                 .sort_values(["rating_count", "avg_rating"], ascending=False))

def recommend_user_based(user_id, user_item_matrix, user_to_idx, idx_to_book,
                         book_title_dict, user_mean, popular_books, k=50, top_n=10):
    if user_id not in user_to_idx:
        top = popular_books.head(top_n).reset_index()
        top["title"] = top["book_id"].map(book_title_dict).fillna("Unknown")
        return top[["book_id", "title", "avg_rating", "rating_count"]]
    u_idx = user_to_idx[user_id]
    user_vector = user_item_matrix[u_idx]

    similarities = cosine_similarity(user_vector, user_item_matrix).flatten()

    similarities[u_idx] = -1

    neighbor_indices = np.argsort(similarities)[::-1][:k]
    neighbor_weights = similarities[neighbor_indices]

    positive_mask = neighbor_weights > 0
    neighbor_indices = neighbor_indices[positive_mask]
    neighbor_weights = neighbor_weights[positive_mask]

    if len(neighbor_indices) == 0:
        top = popular_books.head(top_n).reset_index()
        top["title"] = top["book_id"].map(book_title_dict).fillna("Unknown")
        return top[["book_id", "title", "avg_rating", "rating_count"]]

    neighbor_ratings = user_item_matrix[neighbor_indices]

    weighted_sum = neighbor_ratings.T @ neighbor_weights

    sum_weights = neighbor_weights.sum()

    pred_centered = weighted_sum / sum_weights

    pred_ratings = pred_centered + user_mean[user_id]

    user_rated = user_item_matrix[u_idx].nonzero()[1]
    pred_ratings[user_rated] = -np.inf

    top_item_indices = np.argsort(pred_ratings)[::-1][:top_n]
    top_book_ids = [idx_to_book[i] for i in top_item_indices]
    top_scores = pred_ratings[top_item_indices]

    result = pd.DataFrame({
        "book_id": top_book_ids,
        "predicted_rating": top_scores
    })
    result["title"] = result["book_id"].map(book_title_dict).fillna("Unknown")
    return result[["book_id", "title", "predicted_rating"]]

In [47]:
user_id = 100
recommendations = recommend_user_based(user_id=user_id, user_item_matrix=user_item_matrix, user_to_idx=user_to_idx,
                                       idx_to_book=idx_to_book, book_title_dict=book_title_dict, user_mean=user_mean,
                                       popular_books=popular_books, k=50, top_n=10)

print("\nРекомендации для пользователя (user-based)")
print(recommendations)


Рекомендации для пользователя (user-based)
   book_id                                              title  \
0     4068              Daddy-Long-Legs (Daddy-Long-Legs, #1)   
1     2336                                  The Bhagavad Gita   
2     7208        Cat Among the Pigeons (Hercule Poirot, #32)   
3     9606                Innocent Erendira and Other Stories   
4     8555                      Tintin in Tibet (Tintin, #20)   
5     8403  The Story of Philosophy: The Lives and Opinion...   
6     7269  Arch of Triumph: A Novel of a Man Without a Co...   
7     7920  Great by Choice: Uncertainty, Chaos, and Luck-...   
8     6392                                  The Twelve Chairs   
9     8209                                      رجال في الشمس   

   predicted_rating  
0          4.348166  
1          4.348166  
2          4.348166  
3          4.347850  
4          4.346493  
5          4.345594  
6          4.345157  
7          4.345157  
8          4.343914  
9          4.343914 

item-based подход

In [48]:
title_col = 'title' if 'title' in df_books.columns else 'original_title'
df_books["display_title"] = df_books[title_col].fillna("Unknown")
book_title_dict = df_books.set_index("id")["display_title"].to_dict()
for book_id in df_ratings["book_id"].unique():
    if book_id not in book_title_dict:
        book_title_dict[book_id] = "Unknown"

In [49]:
min_ratings_per_user = 2
user_counts = df_ratings.groupby("user_id").size()
active_users = user_counts[user_counts >= min_ratings_per_user].index
ratings_filtered = df_ratings[df_ratings["user_id"].isin(active_users)].copy()

min_ratings_per_book = 10
book_counts = ratings_filtered.groupby("book_id").size()
popular_books_filter = book_counts[book_counts >= min_ratings_per_book].index
ratings_filtered = ratings_filtered[ratings_filtered["book_id"].isin(popular_books_filter)]

print(f"Пользователей после фильтрации: {ratings_filtered['user_id'].nunique()}")
print(f"Книг после фильтрации: {ratings_filtered['book_id'].nunique()}")

Пользователей после фильтрации: 53424
Книг после фильтрации: 9999


In [50]:
user_mean = ratings_filtered.groupby("user_id")["rating"].mean()
ratings_filtered["rating_centered"] = ratings_filtered["rating"] - ratings_filtered["user_id"].map(user_mean)

book_ids = ratings_filtered["book_id"].unique()
user_ids = ratings_filtered["user_id"].unique()

book_to_idx = {book: i for i, book in enumerate(book_ids)}
user_to_idx = {user: j for j, user in enumerate(user_ids)}
idx_to_book = {i: book for book, i in book_to_idx.items()}
idx_to_user = {j: user for user, j in user_to_idx.items()}

rows = ratings_filtered["book_id"].map(book_to_idx)
cols = ratings_filtered["user_id"].map(user_to_idx)
data = ratings_filtered["rating_centered"].astype(np.float32)

item_user_matrix = csr_matrix(
    (data, (rows, cols)),
    shape=(len(book_ids), len(user_ids)),
    dtype=np.float32
)
print(f"Матрица размером {item_user_matrix.shape}, ненулевых: {item_user_matrix.nnz}")

item_similarity = cosine_similarity(item_user_matrix, dense_output=False)

if hasattr(item_similarity, "toarray"):
    item_similarity = item_similarity.toarray()
print(f"Матрица схожести размером {item_similarity.shape}")

Матрица размером (9999, 53424), ненулевых: 979470
Матрица схожести размером (9999, 9999)


In [51]:
popular_books = (df_ratings.groupby("book_id").agg(avg_rating=("rating", "mean"), rating_count=("rating", "count"))
                 .sort_values(["rating_count", "avg_rating"], ascending=False))

def recommend_item_based(user_id, item_user_matrix, item_similarity, book_to_idx, idx_to_book, book_title_dict,
                         user_to_idx, user_mean, popular_books, top_n=10):
    if user_id not in user_to_idx:
        top = popular_books.head(top_n).reset_index()
        top["title"] = top["book_id"].map(book_title_dict).fillna("Unknown")
        return top[["book_id", "title", "avg_rating", "rating_count"]]

    u_idx = user_to_idx[user_id]

    user_ratings_col = item_user_matrix[:, u_idx]
    rated_book_indices = user_ratings_col.nonzero()[0]
    rated_centered = user_ratings_col[rated_book_indices].toarray().flatten()

    if len(rated_book_indices) == 0:
        top = popular_books.head(top_n).reset_index()
        top["title"] = top["book_id"].map(book_title_dict).fillna("Unknown")
        return top[["book_id", "title", "avg_rating", "rating_count"]]

    sim_to_rated = item_similarity[:, rated_book_indices]
    weighted_sum = sim_to_rated @ rated_centered
    sum_weights = np.abs(sim_to_rated).sum(axis=1)

    sum_weights[sum_weights == 0] = 1e-9
    pred_centered = weighted_sum / sum_weights

    pred_ratings = pred_centered + user_mean[user_id]

    pred_ratings[rated_book_indices] = -np.inf

    top_indices = np.argsort(pred_ratings)[::-1][:top_n]
    top_book_ids = [idx_to_book[i] for i in top_indices]
    top_scores = pred_ratings[top_indices]

    result = pd.DataFrame({
        "book_id": top_book_ids,
        "predicted_rating": top_scores
    })
    result["title"] = result["book_id"].map(book_title_dict).fillna("Unknown")
    return result[["book_id", "title", "predicted_rating"]]

In [52]:
user_id = 100
recommendations = recommend_item_based(user_id=user_id, item_user_matrix=item_user_matrix, item_similarity=item_similarity,
                                       book_to_idx=book_to_idx, idx_to_book=idx_to_book, book_title_dict=book_title_dict,
                                       user_to_idx=user_to_idx, user_mean=user_mean, popular_books=popular_books, top_n=10)

print("\nРекомендации для пользователя (item-based)")
print(recommendations)


Рекомендации для пользователя (item-based)
   book_id                                       title  predicted_rating
0     3142                                Theodore Rex               5.0
1      274                               The Godfather               5.0
2     5556                                     The Sea               5.0
3     6867                                Malgudi Days               5.0
4     5061                           A Matter of Honor               5.0
5     9173  The Twentieth Wife (Taj Mahal Trilogy, #1)               5.0
6     7359                                  Swing Time               5.0
7     7837                        Twilight and History               5.0
8     8326                               The Enchanted               5.0
9      130                     The Old Man and the Sea               5.0
